# Creating a Poisson Model to try and predict the score of a Premier League Game

Our target here is FTHG and FTAG

In [69]:
import pandas as pd
import numpy as np
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance, mean_absolute_error

In [2]:
url = "https://raw.githubusercontent.com/hankike/Premier_Prediction/refs/heads/main/data/cleaned_premier.csv"

df_premier = pd.read_csv(url)

In [3]:
# Sort original match dataframe chronologically
df_premier = df_premier.sort_values(
    ['Date', 'Time']
).reset_index(drop=True)

# Total goals in each match
df_premier['Total_Goals'] = (
    df_premier['FTHG'] + df_premier['FTAG']
)

# Cumulative league goals before each match
df_premier['League_Total_Goals'] = (
    df_premier['Total_Goals'].shift(1).cumsum()
)

# Number of previous matches
df_premier['League_Total_Matches'] = (
    df_premier['Total_Goals'].shift(1).expanding().count()
)

# Adding match ID
df_premier['Match_ID'] = range(len(df_premier))

# Average goals per team per match
df_premier['League_Avg_Goals'] = (
    df_premier['League_Total_Goals'] /
    (df_premier['League_Total_Matches'] * 2)
)

First I wish to update this data set in a way that makes every game into two rows, one for each team

In [5]:
# Home team
home_df = df_premier[
    ['Match_ID', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']
].copy()

home_df = home_df.rename(columns={
    'HomeTeam': 'Team',
    'AwayTeam': 'Opponent',
    'FTHG': 'Goals_Scored',
    'FTAG': 'Goals_Allowed'
})

home_df['Home'] = 1

# Away team
away_df = df_premier[
    ['Match_ID', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']
].copy()

away_df = away_df.rename(columns={
    'AwayTeam': 'Team',
    'HomeTeam': 'Opponent',
    'FTAG': 'Goals_Scored',
    'FTHG': 'Goals_Allowed'
})

away_df['Home'] = 0

# Combine
team_df = pd.concat([home_df, away_df], ignore_index=True)

# Convert the date
team_df['Date'] = pd.to_datetime(team_df['Date'], dayfirst=True)

# Sort
team_df = team_df.sort_values(['Date', 'Team']).reset_index(drop=True)

In [6]:
team_df = team_df.merge(
    df_premier[['Match_ID', 'League_Avg_Goals']],
    on='Match_ID',
    how='left'
)

In [7]:
# Create our points based on wins and draws
team_df['Points'] = np.select(
    [
        team_df['Goals_Scored'] > team_df['Goals_Allowed'],
        team_df['Goals_Scored'] == team_df['Goals_Allowed']
    ],
    [3, 1],
    default=0
)

# Count number of wins
team_df['Win'] = (
    team_df['Goals_Scored'] > team_df['Goals_Allowed']
).astype(int)

### Rolling Features

Next, I wish to capture how the teams are performing based on their attack, defense, and results.

To do this I am calculating how the teams have been performing recently using rolling features. This is an essential aspect of prediction modeling as you can never use future outcomes when finding your prediction values.

In [8]:
# Group by team
grouped = team_df.groupby('Team')

# Shift by 1 to exclude the current match
team_df['Avg_Goals_Scored_5'] = (
    grouped['Goals_Scored']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_df['Avg_Goals_Scored_10'] = (
    grouped['Goals_Scored']
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

team_df['Avg_Goals_Allowed_5'] = (
    grouped['Goals_Allowed']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_df['Avg_Goals_Allowed_10'] = (
    grouped['Goals_Allowed']
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

team_df['Points_Last_5'] = (
    grouped['Points']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

team_df['Wins_Last_5'] = (
    grouped['Win']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

### Offensive and Defensive Rankings

We wish to find metrics of how well a team is performing compared to league average

In [9]:
team_df['Attack_Strength'] = (
    team_df['Avg_Goals_Scored_10'] / team_df['League_Avg_Goals']
)

team_df['Defense_Strength'] = (
    team_df['Avg_Goals_Allowed_10'] / team_df['League_Avg_Goals']
)

# Now let's predict home team goals and away team goals

In [11]:
home_features = team_df[team_df['Home'] == 1].copy()
away_features = team_df[team_df['Home'] == 0].copy()

In [12]:
home_features = home_features.rename(columns={
    'Team': 'HomeTeam',
    'Opponent': 'AwayTeam',
    'Goals_Scored': 'FTHG',
    'Goals_Allowed': 'FTAG',
    'Attack_Strength': 'Home_Attack',
    'Defense_Strength': 'Home_Defense',
    'Avg_Goals_Scored_5': 'Home_Avg_Goals_5',
    'Avg_Goals_Scored_10': 'Home_Avg_Goals_10',
    'Avg_Goals_Allowed_5': 'Home_Avg_Allowed_5',
    'Avg_Goals_Allowed_10': 'Home_Avg_Allowed_10',
    'Points_Last_5': 'Home_Points_5',
    'Wins_Last_5': 'Home_Wins_5'
})

away_features = away_features.rename(columns={
    'Team': 'AwayTeam',
    'Opponent': 'HomeTeam',
    'Goals_Scored': 'FTAG',
    'Goals_Allowed': 'FTHG',
    'Attack_Strength': 'Away_Attack',
    'Defense_Strength': 'Away_Defense',
    'Avg_Goals_Scored_5': 'Away_Avg_Goals_5',
    'Avg_Goals_Scored_10': 'Away_Avg_Goals_10',
    'Avg_Goals_Allowed_5': 'Away_Avg_Allowed_5',
    'Avg_Goals_Allowed_10': 'Away_Avg_Allowed_10',
    'Points_Last_5': 'Away_Points_5',
    'Wins_Last_5': 'Away_Wins_5'
})

Build the models

In [14]:
home_features_list = [
    'Home_Attack',
    'Away_Defense',
    'Home_Avg_Goals_5',
    'Home_Avg_Goals_10',
    'Away_Avg_Allowed_5',
    'Away_Avg_Allowed_10',
    'Home_Points_5',
    'Home_Wins_5'
]

away_features_list = [
    'Away_Attack',
    'Home_Defense',
    'Away_Avg_Goals_5',
    'Away_Avg_Goals_10',
    'Home_Avg_Allowed_5',
    'Home_Avg_Allowed_10',
    'Away_Points_5',
    'Away_Wins_5'
]

In [16]:
match_df = home_features.merge(
    away_features,
    on=['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG'],
    suffixes=('_home', '_away')
)

train / test

In [17]:
match_df = match_df.sort_values('Date').reset_index(drop=True)

split_index = int(len(match_df) * 0.8)

train_df = match_df.iloc[:split_index]
test_df = match_df.iloc[split_index:]

In [20]:
train_df[train_df[home_features_list].isna().any(axis=1)]

,Match_ID_home,Date,Time_home,HomeTeam,AwayTeam,FTHG,FTAG,Home_home,League_Avg_Goals_home,Points_home,...,Points_away,Win_away,Away_Avg_Goals_5,Away_Avg_Goals_10,Away_Avg_Allowed_5,Away_Avg_Allowed_10,Away_Points_5,Away_Wins_5,Away_Attack,Away_Defense
0,768,2019-08-09,20:00,Liverpool,Norwich,4,1,1,1.480469,3,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,842,2019-08-10,15:00,Bournemouth,Sheffield United,1,1,1,1.475653,1,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,843,2019-08-10,15:00,Burnley,Southampton,3,0,1,1.475089,3,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,844,2019-08-10,15:00,Crystal Palace,Everton,0,0,1,1.475118,1,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,846,2019-08-10,17:30,Tottenham,Aston Villa,3,1,1,1.473404,3,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,845,2019-08-10,15:00,Watford,Brighton,0,3,1,1.473373,0,...,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,841,2019-08-10,12:30,West Ham,Man City,0,5,1,1.474435,0,...,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,922,2019-08-11,14:00,Leicester,Wolves,0,0,1,1.476139,1,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,924,2019-08-11,16:30,Man United,Chelsea,4,0,1,1.473485,3,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,923,2019-08-11,14:00,Newcastle,Arsenal,0,1,1,1.474540,0,...,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
train_df = train_df.dropna(
    subset=home_features_list + away_features_list
)

test_df = test_df.dropna(
    subset=home_features_list + away_features_list
)

In [25]:
home_model = PoissonRegressor(alpha=1.0, max_iter=1000)
away_model = PoissonRegressor(alpha=1.0, max_iter=1000)

home_model.fit(
    train_df[home_features_list],
    train_df['FTHG']
)

away_model.fit(
    train_df[away_features_list],
    train_df['FTAG']
)

,"max_iter max_iter: int, default=100The maximal number of iterations for the solver.Values must be in the range `[1, inf)`.",1000
,"alpha alpha: float, default=1Constant that multiplies the L2 penalty term and determines theregularization strength. ``alpha = 0`` is equivalent to unpenalizedGLMs. In this case, the design matrix `X` must have full column rank(no collinearities).Values of `alpha` must be in the range `[0.0, inf)`.",1.0
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the linear predictor (`X @ coef + intercept`).",True
,"solver solver: {'lbfgs', 'newton-cholesky'}, default='lbfgs'Algorithm to use in the optimization problem:'lbfgs' Calls scipy's L-BFGS-B optimizer.'newton-cholesky' Uses Newton-Raphson steps (in arbitrary precision arithmetic equivalent to iterated reweighted least squares) with an inner Cholesky based solver. This solver is a good choice for `n_samples` >> `n_features`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features` because it explicitly computes the Hessian matrix. .. versionadded:: 1.2",'lbfgs'
,"tol tol: float, default=1e-4Stopping criterion. For the lbfgs solver,the iteration will stop when ``max{|g_j|, j = 1, ..., d} <= tol``where ``g_j`` is the j-th component of the gradient (derivative) ofthe objective function.Values must be in the range `(0.0, inf)`.",0.0001
,"warm_start warm_start: bool, default=FalseIf set to ``True``, reuse the solution of the previous call to ``fit``as initialization for ``coef_`` and ``intercept_`` .",False
,"verbose verbose: int, default=0For the lbfgs solver set verbose to any positive number for verbosity.Values must be in the range `[0, inf)`.",0
Name,Type,Value
"coef_ coef_: array of shape (n_features,)Estimated coefficients for the linear predictor (`X @ coef_ +intercept_`) in the GLM.","ndarray[float64](8,)","[0.04,0.04,0.04,...,0.05,0.02,0.01]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['Away_Attack','Home_Defense','Away_Avg_Goals_5',...,'Home_Avg_Allowed_10', 'Away_Points_5','Away_Wins_5']"
intercept_ intercept_: floatIntercept (a.k.a. bias) added to linear predictor.,float64,-0.2586


In [26]:
test_df = test_df.copy()

test_df['Predicted_FTHG'] = home_model.predict(
    test_df[home_features_list]
)

test_df['Predicted_FTAG'] = away_model.predict(
    test_df[away_features_list]
)

In [27]:
print("Home Goals MAE:",
      mean_absolute_error(test_df['FTHG'], test_df['Predicted_FTHG']))

print("Away Goals MAE:",
      mean_absolute_error(test_df['FTAG'], test_df['Predicted_FTAG']))

print("Home Goals Poisson Deviance:",
      mean_poisson_deviance(test_df['FTHG'], test_df['Predicted_FTHG']))

print("Away Goals Poisson Deviance:",
      mean_poisson_deviance(test_df['FTAG'], test_df['Predicted_FTAG']))

Home Goals MAE: 0.9871252758888778
Away Goals MAE: 0.8648782944441162
Home Goals Poisson Deviance: 1.1444353834150038
Away Goals Poisson Deviance: 1.080469151336184


Now we have a Poisson value

Let us try and build a prediction model from it and find scores